# 🚀 Aelionix Academy — AI Tutor QLoRA Fine-Tuning & Live Tunnel

This notebook provides an automated, **100% zero-cost** workflow to:
1. **Fine-tune Qwen 2.5 Coder 7B** with 4-bit QLoRA on a free Google Colab Tesla T4 GPU.
2. **Host live OpenAI-compatible inference** directly from Google Colab.
3. **Expose a public HTTPS tunnel** via Cloudflare Tunnel for **$0**, connecting live to your local Aelionix Academy app (`http://localhost:3000`).
4. **(Phase 2)** Push the trained adapter to Hugging Face Model Hub for serverless deployment on Together AI when launching to live users.

### ⚙️ Step 0: Ensure GPU is Enabled in Colab
In the top menu, go to: **Runtime** -> **Change runtime type** -> select **T4 GPU** -> click **Save**.

## Step 1: Verify Hardware Accelerator (Tesla T4 GPU)

In [ ]:
!nvidia-smi

## Step 2: Install Fine-Tuning & API Serving Dependencies
This installs PyTorch quantization tools (`bitsandbytes`, `peft`), Hugging Face `transformers`, `datasets`, and FastAPI/Uvicorn for the inference endpoint.

In [ ]:
!pip install -q --upgrade pip
!pip install -q transformers accelerate bitsandbytes peft datasets fastapi uvicorn

## Step 3: Fetch Dataset Files (No Repo Clone Needed)
The notebook only needs the two dataset files: `train.jsonl` and `eval.jsonl` (~190 KB total).
You do **not** need to clone the entire web application, frontend components, or node modules.

- **Method A (Auto-Fetch)**: Set your GitHub `username/repo` below to download only the dataset files in 1 second.
- **Method B (File Upload)**: Leave it empty, and Colab will present an upload button to pick the two files directly from your computer.

In [ ]:
import os
import urllib.request
from pathlib import Path
from google.colab import files

# Default GitHub repository to auto-fetch dataset splits
GITHUB_REPO = "sagelord01/Aelionix_academy"
BRANCH = "main"

os.makedirs("data/traces/splits", exist_ok=True)
train_dest = Path("data/traces/splits/train.jsonl")
eval_dest = Path("data/traces/splits/eval.jsonl")

fetched = False
if GITHUB_REPO.strip():
    try:
        print(f"Fetching dataset files from GitHub: {GITHUB_REPO} (branch: {BRANCH})...")
        train_url = f"https://raw.githubusercontent.com/{GITHUB_REPO.strip()}/{BRANCH}/data/traces/splits/train.jsonl"
        eval_url = f"https://raw.githubusercontent.com/{GITHUB_REPO.strip()}/{BRANCH}/data/traces/splits/eval.jsonl"
        urllib.request.urlretrieve(train_url, str(train_dest))
        urllib.request.urlretrieve(eval_url, str(eval_dest))
        print("✓ Successfully fetched train.jsonl and eval.jsonl from GitHub without cloning the repo!")
        fetched = True
    except Exception as e:
        print(f"Could not auto-fetch from GitHub ({e}). Falling back to local upload...")

if not fetched and not (train_dest.exists() and eval_dest.exists()):
    print("\nPlease upload 'train.jsonl' and 'eval.jsonl' from your computer's 'data/traces/splits/' folder:")
    uploaded = files.upload()
    for filename in uploaded.keys():
        dest = Path("data/traces/splits") / filename
        if dest.exists():
            dest.unlink()
        os.rename(filename, str(dest))
    print("\n✓ Dataset splits loaded successfully!")

# Quick validation check
with open(train_dest, "r", encoding="utf-8") as f:
    train_count = sum(1 for _ in f)
with open(eval_dest, "r", encoding="utf-8") as f:
    eval_count = sum(1 for _ in f)

print(f"\n✓ Ready: {train_count} train examples and {eval_count} eval examples loaded into data/traces/splits/")

## Step 4: Fine-Tune Qwen 2.5 Coder 7B with QLoRA
- **Base Model**: `Qwen/Qwen2.5-Coder-7B-Instruct`
- **Quantization**: 4-bit NF4 (fits cleanly within free 16 GB VRAM)
- **Loss Masking**: Calculates loss **strictly** on the tutor's JSON responses (prevents wasting model capacity memorizing system prompts)
- **Runtime**: ~8-12 minutes on free T4

In [ ]:
import json
import os
import sys
from pathlib import Path
import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_LENGTH = 2048
EPOCHS = 3
OUTPUT_DIR = Path("aelionix-tutor-qlora")
train_path = Path("data/traces/splits/train.jsonl")
eval_path = Path("data/traces/splits/eval.jsonl")

print("=" * 60)
print(f"Training Target: {MODEL_NAME}")
print(f"CUDA Device    : {torch.cuda.get_device_name(0)}")
print(f"VRAM Available : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("=" * 60)

# 1. Load Tokenizer
print("\n[1/5] Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. Load Base Model in 4-bit NF4
print("[2/5] Loading base model in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.config.use_cache = False

# 3. Attach LoRA Adapter
print("[3/5] Attaching LoRA adapter...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# 4. Tokenize and Mask Prompt Loss
print("[4/5] Tokenizing dataset and applying prompt loss masking...")
dataset = load_dataset("json", data_files={"train": str(train_path), "eval": str(eval_path)})

def tokenize(example):
    messages = example["messages"]
    prompt_text = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LENGTH)
    input_ids = full["input_ids"]
    labels = list(input_ids)
    mask_len = min(len(prompt_ids), len(labels))
    for i in range(mask_len):
        labels[i] = -100
    if all(l == -100 for l in labels):
        labels[-1] = input_ids[-1]
    return {"input_ids": input_ids, "attention_mask": full["attention_mask"], "labels": labels}

tokenized = dataset.map(tokenize, remove_columns=dataset["train"].column_names, desc="Tokenizing")

# 5. Train with FP16 and Gradient Checkpointing
print("[5/5] Launching Trainer...")
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    optim="paged_adamw_32bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    remove_unused_columns=False,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["eval"],
    data_collator=collator,
)

print("\n🔥 Training begins! This will take ~8-12 minutes on T4...")
trainer.train()

print("\n💾 Saving fine-tuned adapter...")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"🎉 Adapter saved successfully in: {OUTPUT_DIR}/")

## Step 5: Start Live OpenAI-Compatible API + Cloudflare Tunnel (Developer Testing)
This cell starts an OpenAI-compatible FastAPI server and launches a free Cloudflare Tunnel.

**Keep this cell running while testing on your local machine!**
Copy the generated `https://*.trycloudflare.com/v1` URL into your local `.env.local` file and refresh your app.

In [ ]:
import os
import re
import subprocess
import threading
import time
from typing import Any, Optional
from pathlib import Path
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, ConfigDict
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Ensure cloudflared is installed
if subprocess.call(["which", "cloudflared"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL) != 0:
    print("Installing cloudflared binary...")
    os.system("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb")
    os.system("dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1")

# 2. Initialize FastAPI app
app = FastAPI(title="Aelionix Tutor OpenAI-Compatible API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatMessage(BaseModel):
    model_config = ConfigDict(extra="allow")
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    model_config = ConfigDict(extra="allow")
    model: str = "aelionix-tutor"
    messages: list[ChatMessage]
    temperature: float = 0.45
    max_tokens: int = 1200
    response_format: Optional[Any] = None

print("Loading fine-tuned model and adapter for serving...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
serve_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
if serve_tokenizer.pad_token is None:
    serve_tokenizer.pad_token = serve_tokenizer.eos_token

base_serve_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

adapter_path = Path("aelionix-tutor-qlora")
if adapter_path.exists():
    print(f"Attaching fine-tuned adapter from {adapter_path}...")
    serve_model = PeftModel.from_pretrained(base_serve_model, str(adapter_path))
else:
    print("Warning: Adapter path not found. Serving base model.")
    serve_model = base_serve_model

serve_model.eval()
print("✓ Model loaded into GPU memory!")

@app.get("/health")
def health():
    return {"status": "ok", "model": "aelionix-tutor"}

@app.post("/v1/chat/completions")
@app.post("/chat/completions")
def chat_completions(req: ChatCompletionRequest):
    formatted = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt = serve_tokenizer.apply_chat_template(
        formatted,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = serve_tokenizer(prompt, return_tensors="pt").to(serve_model.device)
    
    with torch.no_grad():
        output_ids = serve_model.generate(
            **inputs,
            max_new_tokens=min(req.max_tokens, 450),
            temperature=req.temperature,
            do_sample=req.temperature > 0.05,
            pad_token_id=serve_tokenizer.pad_token_id,
            eos_token_id=serve_tokenizer.eos_token_id,
            use_cache=True,
        )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    reply = serve_tokenizer.decode(generated, skip_special_tokens=True).strip()
    
    return {
        "id": f"chatcmpl-aelionix-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [
            {
                "index": 0,
                "message": {"role": "assistant", "content": reply},
                "finish_reason": "stop",
            }
        ],
        "usage": {
            "prompt_tokens": inputs.input_ids.shape[1],
            "completion_tokens": len(generated),
            "total_tokens": inputs.input_ids.shape[1] + len(generated),
        },
    }

def launch_cloudflare_tunnel():
    print("\nStarting Cloudflare Tunnel on port 8000...")
    log_path = Path("/content/tunnel.log") if Path("/content").exists() else Path("tunnel.log")
    if log_path.exists():
        log_path.unlink()
    os.system(f"nohup cloudflared tunnel --url http://127.0.0.1:8000 --logfile {log_path} > /dev/null 2>&1 &")
    tunnel_url = None
    for _ in range(25):
        time.sleep(1)
        if log_path.exists():
            content = log_path.read_text(encoding="utf-8", errors="ignore")
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                tunnel_url = match.group(0)
                break
    
    if tunnel_url:
        print("\n" + "=" * 75)
        print("🎉 AELIONIX TUTOR LIVE INFERENCE SERVER IS READY!")
        print("=" * 75)
        print(f"Public API Base URL:  {tunnel_url}/v1")
        print("\nCopy and paste these 5 lines into your local .env.local:")
        print("---------------------------------------------------------------------------")
        print("AI_PROVIDER=openai-compatible")
        print(f"AI_BASE_URL={tunnel_url}/v1")
        print("AI_MODEL=aelionix-tutor")
        print("AI_API_KEY=colab-tunnel")
        print("AI_SUPPORTS_JSON_MODE=true")
        print("=" * 75 + "\n")
    else:
        print("Could not extract tunnel URL. Please check cloudflared process.")

# 1. Start FastAPI server in background thread with its own asyncio loop
def start_api():
    import asyncio
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

threading.Thread(target=start_api, daemon=True).start()

# 2. Launch Cloudflare Tunnel and wait for public HTTPS URL
launch_cloudflare_tunnel()

## Step 6 (Phase 2): Push Adapter to Hugging Face Hub (Production Launch)
When you finish developer testing and are happy with the tutor's answers, run this cell.
It uploads your trained adapter weights (~100 MB) to Hugging Face Model Hub (free permanent storage).

From there, you can connect your adapter to **Together AI** serverless inference ($5 free credits = 25,000+ live requests with $0 idle cost) for production learners!

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login

# 1. Paste your Hugging Face write token from: https://huggingface.co/settings/tokens
print("Enter your Hugging Face WRITE token when prompted:")
login()

# 2. Specify your Hugging Face username
HF_USERNAME = "your-username"  # <-- Replace with your Hugging Face username
HF_REPO_NAME = f"{HF_USERNAME}/aelionix-tutor-qlora"

print(f"\nPushing adapter to https://huggingface.co/{HF_REPO_NAME}...")
serve_model.push_to_hub(HF_REPO_NAME)
serve_tokenizer.push_to_hub(HF_REPO_NAME)

print("\n" + "=" * 75)
print(f"🎉 SUCCESS! Adapter published to: https://huggingface.co/{HF_REPO_NAME}")
print("You are now ready to link this repo in Together AI for serverless user testing!")
print("=" * 75)